# Course 6 lab — Policy-as-Code and runtime governance

**Scenario:** A procurement agent proposes purchase orders, but a trusted application resolves facts, a versioned policy bundle decides, and a runtime gateway alone can apply effects.

You will compare an unsafe baseline with a composed policy, validate and mutation-test a bundle, shadow and canary a candidate, exercise rollback, and verify a non-bypassable, idempotent effect path. The lab is deterministic, credential-free, and imports the same implementation exercised by the course tests.


## 1. Load the tested lab

The notebook does not redefine its own policy engine. It loads `lab.py`, the canonical implementation used by `tests/test_module06_runtime_policy.py`. No package installation or network service is required.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import importlib.util
import sys

course_dir = Path("curriculum/intermediate/06-policy-as-code-and-runtime-governance").resolve()
spec = importlib.util.spec_from_file_location("course06_lab", course_dir / "lab.py")
lab = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)

assert lab.validate_bundle(lab.stable_bundle()).valid
print("Loaded tested Course 6 lab:", lab.stable_bundle().version)


## 2. Establish the unsafe baseline

The baseline trusts a coarse workload prefix and returns on the first matching allow. That design skips sanctions, hard limits, safety constraints, and escalation. The labelled corpus defines exactly 2 legitimate allows, 6 hard denials, and 2 escalations.


In [ ]:
cases = lab.labelled_cases()
summary = lab.summarize_evaluation(lab.stable_bundle(), cases)
summary.model_dump()


In [ ]:
assert summary.case_count == 10
assert summary.baseline_correct_count == 2
assert summary.baseline_forbidden_allowed_count == 6
assert summary.baseline_missed_escalation_count == 2
assert summary.candidate_correct_count == 10
assert summary.candidate_forbidden_allowed_count == 0
print("The composed candidate removes all labelled baseline safety errors.")


## 3. Resolve trusted inputs, then compose policy domains

The proposal may contain model claims, but policy facts come from authoritative services. Every required domain is evaluated. Composition is explicit: **hard DENY > ESCALATE > ALLOW**.


In [ ]:
context = lab.demo_context()
proposal = lab.demo_proposal(model_claimed_approval=True, model_claimed_vendor_safe=True)
facts = lab.demo_facts(proposal, vendor_sanctioned=True)
decision = lab.evaluate_policy(
    lab.stable_bundle(), context, proposal, facts, now=lab.REFERENCE_TIME
)
[(item.domain, item.outcome.value, item.reason_codes) for item in decision.domain_decisions]


In [ ]:
assert decision.outcome is lab.Outcome.DENY
assert "vendor_sanctioned" in decision.reason_codes
assert decision.bundle_digest == lab.stable_bundle().digest
print("Untrusted model claims did not override authoritative vendor facts.")


## 4. Inject runtime failures

Missing, stale, mismatched, and cross-tenant facts must not quietly become an allow. For state-changing actions this lab fails closed when the policy control plane is unavailable.


In [ ]:
safe_proposal = lab.demo_proposal()
missing = lab.evaluate_policy(
    lab.stable_bundle(), context, safe_proposal, None, now=lab.REFERENCE_TIME
)
other = lab.demo_proposal(operation_id="OP-OTHER")
mismatch = lab.evaluate_policy(
    lab.stable_bundle(), context, safe_proposal, lab.demo_facts(other), now=lab.REFERENCE_TIME
)
assert missing.outcome is lab.Outcome.DENY
assert mismatch.outcome is lab.Outcome.DENY
print(missing.reason_codes, mismatch.reason_codes)


In [ ]:
control, gateway, adapter = lab.build_demo_gateway()
control.set_available(False)
outage = gateway.execute(
    context, safe_proposal, lab.demo_facts(safe_proposal), now=lab.REFERENCE_TIME
)
assert outage.decision.enforced.outcome is lab.Outcome.DENY
assert outage.effect.status is lab.EffectStatus.NOT_ATTEMPTED
assert adapter.applied_count == 0
outage.decision.enforced.reason_codes


## 5. Validate a policy release

Syntax is only the first gate. This validator checks the input schema version, complete domain coverage, and satisfiable amount/risk thresholds. Production engines add language-specific parsing, type checking, schema validation, linting, and automated reasoning.


In [ ]:
invalid = lab.stable_bundle().model_copy(update={
    "version": "invalid-1",
    "required_domains": frozenset({"authorization"}),
    "rules": lab.PolicyRules(
        schema_version="wrong/v2",
        approval_threshold_cents=3_000_000,
        hard_amount_limit_cents=2_000_000,
        risk_escalation_basis_points=9_000,
        risk_deny_basis_points=8_000,
    ),
})
report = lab.validate_bundle(invalid)
assert not report.valid
[item.code for item in report.findings]


## 6. Kill a policy mutation

A candidate widens the hard amount limit. Static validation alone accepts the internally consistent thresholds, but the semantic corpus catches a forbidden transaction changing from DENY to ESCALATE. For this course, any non-DENY result on a labelled hard-deny case is a false permit.


In [ ]:
active = lab.stable_bundle()
mutant = lab.mutated_bundle()
assert lab.validate_bundle(mutant).valid
mutation_metrics = lab.shadow_metrics(active, mutant, cases)
assert mutation_metrics.false_permit_count == 0
assert mutation_metrics.forbidden_not_denied_count == 1
mutation_metrics.model_dump()


## 7. Shadow before enforcement

A stricter candidate is evaluated against the same request while the active bundle remains authoritative. Shadow output is evidence, never an enforcement result.


In [ ]:
control, gateway, adapter = lab.build_demo_gateway(active)
strict_candidate = active.model_copy(update={
    "version": "procurement-policy-1.1.0",
    "revision": "git:2b7c9e1",
    "stage": lab.ReleaseStage.DRAFT,
    "rules": active.rules.model_copy(update={"approval_threshold_cents": 100_000}),
})
assert control.register(strict_candidate).valid
control.start_shadow(strict_candidate.version)
proposal = lab.demo_proposal()
shadowed = gateway.execute(
    context, proposal, lab.demo_facts(proposal), now=lab.REFERENCE_TIME
)
assert shadowed.decision.enforced.outcome is lab.Outcome.ALLOW
assert shadowed.decision.shadow.outcome is lab.Outcome.ESCALATE
assert shadowed.effect.status is lab.EffectStatus.APPLIED
{
    "enforced": shadowed.decision.enforced.outcome.value,
    "shadow": shadowed.decision.shadow.outcome.value,
    "enforced_version": shadowed.decision.enforced.bundle_version,
    "candidate_version": shadowed.decision.shadow.bundle_version,
}


## 8. Canary assignment is deterministic

The teaching control plane uses a stable operation-ID hash for cohort assignment. Real systems also need identity-aware cohort design, capacity controls, monitoring windows, and an emergency rollback owner.


In [ ]:
control.start_canary(strict_candidate.version, 10)
canary_operation = next(
    f"OP-CANARY-{i}" for i in range(10_000)
    if lab._in_canary(f"OP-CANARY-{i}", 10)
)
canary_proposal = lab.demo_proposal(operation_id=canary_operation)
canary_facts = lab.demo_facts(canary_proposal)
canary_a = gateway.decide(context, canary_proposal, canary_facts, now=lab.REFERENCE_TIME)
canary_b = gateway.decide(context, canary_proposal, canary_facts, now=lab.REFERENCE_TIME)
assert canary_a.enforced == canary_b.enforced
assert canary_a.enforced.bundle_version == strict_candidate.version
canary_a.enforced.outcome.value


## 9. Release gates and rollback

A candidate with zero labelled safety regressions can be promoted. The registry records an immutable version/digest and can atomically restore a known-good release.


In [ ]:
safe_candidate = active.model_copy(update={
    "version": "procurement-policy-1.0.1",
    "revision": "git:74e8aa0",
    "stage": lab.ReleaseStage.DRAFT,
})
release_control = lab.PolicyControlPlane(active)
assert release_control.register(safe_candidate).valid
release_control.start_shadow(safe_candidate.version)
safe_metrics = lab.shadow_metrics(active, safe_candidate, cases)
assert safe_metrics.incorrect_case_count == 0
assert safe_metrics.false_permit_count == 0
assert safe_metrics.missed_escalation_count == 0
assert safe_metrics.candidate_bundle_digest == safe_candidate.digest
assert safe_metrics.evaluation_corpus_digest == lab.stable_digest(cases)
release_control.start_canary(safe_candidate.version, 10)
release_control.promote(safe_candidate.version, safe_metrics)
assert release_control.snapshot()[0].version == safe_candidate.version
release_control.rollback(active.version)
assert release_control.snapshot()[0].version == active.version
print("Promoted, then restored:", active.version)


## 10. Enforce through one effect boundary

The adapter rejects direct calls. The gateway evaluates immediately before the effect, records the policy/evidence versions, and makes retries idempotent. It never substitutes an `executed=True` flag for an actual effect receipt.


In [ ]:
control, gateway, adapter = lab.build_demo_gateway()
proposal = lab.demo_proposal()
facts = lab.demo_facts(proposal)
first = gateway.execute(context, proposal, facts, now=lab.REFERENCE_TIME)
retry = gateway.execute(context, proposal, facts, now=lab.REFERENCE_TIME)
assert first.effect.status is lab.EffectStatus.APPLIED
assert retry.decision.replayed
assert first.effect == retry.effect
assert adapter.applied_count == 1
try:
    adapter.apply(context.tenant_id, proposal)
except PermissionError as exc:
    bypass_result = str(exc)
else:
    raise AssertionError("direct adapter call unexpectedly succeeded")
{
    "effect": first.effect.model_dump(),
    "evidence_versions": first.decision.evidence_versions,
    "bypass": bypass_result,
}


## 11. Protect tenant and concurrency boundaries

Idempotency is scoped by tenant and logical operation. Eight concurrent retries must produce one target-system effect, while a second tenant using the same operation ID receives a distinct receipt.


In [ ]:
race_context = lab.demo_context()
race_proposal = lab.demo_proposal(operation_id="OP-RACE")
race_facts = lab.demo_facts(race_proposal)
_, race_gateway, race_adapter = lab.build_demo_gateway()
with ThreadPoolExecutor(max_workers=8) as pool:
    race_results = list(pool.map(
        lambda _: race_gateway.execute(
            race_context, race_proposal, race_facts, now=lab.REFERENCE_TIME
        ),
        range(8),
    ))
assert race_adapter.applied_count == 1
assert len({item.effect.effect_id for item in race_results}) == 1
second_tenant = lab.demo_context(tenant_id="tenant:subsidiary")
second_result = race_gateway.execute(
    second_tenant,
    race_proposal,
    lab.demo_facts(race_proposal, tenant_id=second_tenant.tenant_id),
    now=lab.REFERENCE_TIME,
)
assert second_result.effect.effect_id != race_results[0].effect.effect_id
assert race_adapter.applied_count == 2
print("replays:", sum(item.decision.replayed for item in race_results), "effects:", race_adapter.applied_count)


## 12. Preserve unknown effects and minimized evidence

A timeout is not proof that the effect failed. The gateway records the authorized logical operation, returns `UNKNOWN` on retry when no receipt can be reconciled, and does not execute again automatically. Decision evidence stores digests, versions, outcomes, and reason codes—not raw vendor records.


In [ ]:
_, unknown_gateway, unknown_adapter = lab.build_demo_gateway()
unknown_proposal = lab.demo_proposal(operation_id="OP-UNKNOWN")
unknown_facts = lab.demo_facts(unknown_proposal)
original_apply = unknown_adapter.apply
def lose_response(*_args, **_kwargs):
    raise TimeoutError("effect outcome unknown")
unknown_adapter.apply = lose_response
try:
    unknown_gateway.execute(
        context, unknown_proposal, unknown_facts, now=lab.REFERENCE_TIME
    )
except TimeoutError:
    pass
finally:
    unknown_adapter.apply = original_apply
unknown_retry = unknown_gateway.execute(
    context, unknown_proposal, unknown_facts, now=lab.REFERENCE_TIME
)
assert unknown_retry.effect.status is lab.EffectStatus.UNKNOWN
audit_event = gateway.audit_events[0]
assert audit_event.request_digest == first.decision.enforced.input_digest
assert audit_event.bundle_digest == active.digest
assert "vendor:acme" not in audit_event.model_dump_json()
{"retry_status": unknown_retry.effect.status.value, "audit": audit_event.model_dump()}


## 13. Map the invariant to common engines

The canonical lab stays local. `opa_input` shows the exact JSON trust boundary for an OPA adapter; the Rego and Cedar snippets are illustrative production-language artifacts, not claims that those engines executed in this notebook.

- **OPA/Rego:** use `opa check`, `opa test --coverage`, Regal linting, signed/versioned bundles, status, and decision logs.
- **Cedar:** validate policies and requests against a schema; test permit/forbid/default-deny behavior with the Cedar CLI/SDK.
- **Amazon Verified Permissions:** the application remains the PEP and constructs PARC requests from trusted facts.
- **AgentCore Policy:** use policy-level `LOG_ONLY` for shadow testing and `ACTIVE` for enforcement; current Dogwood temporal policies can express session-history constraints.


In [ ]:
payload = lab.opa_input(context, proposal, facts)
assert payload["facts"]["proposal_digest"] == lab.stable_digest(proposal)
assert "default decision" in lab.REGO_ARTIFACT
assert "forbid" in lab.CEDAR_ARTIFACT
print(lab.REGO_ARTIFACT)
print("--- Cedar mapping ---")
print(lab.CEDAR_ARTIFACT)


## 14. Production upgrade exercise

Extend the lab without weakening its invariants:

1. Add a request-bound approval receipt that resolves `ESCALATE` but cannot override a hard DENY.
2. Add a temporal rule: deployment requires a matching test result earlier in the same authenticated session.
3. Replace the local evaluator with an OPA or Cedar adapter while preserving the labelled corpus.
4. Add signed bundle verification, separation of release duties, durable decision logs, and alerts for incomplete shadow evaluation.
5. Define SLOs for decision latency, stale bundle age, missing trusted context, and rollback time.

**Exit check:** explain why model output is a proposal, why shadow decisions cannot enforce, why version/digest evidence matters, and why an effect receipt is stronger than an `executed` Boolean.
